# Web Scraping for Digital Discourse — 2026

**DE0823 (legacy ETH713) · Technical Week 5 · 25 November 2026 · 90 minutes**

**Research question:** How can we turn web documents into a transparent, reproducible dataset for discourse analysis?

Run in Google Colab or local Jupyter. The core notebook defaults to synthetic offline HTML; no third-party website or Google Drive is needed once dependencies are installed. The examples are invented teaching data, not historical quotations or real advertisements. Optional live requests are explicitly enabled by the student.

By the end you should be able to retrieve HTML, explain parsing and document trees, extract text and attributes, create records, export them as JSON/CSV, and interpret a small corpus critically.

Previous notebooks: [Week 3 — CountingWords_2026](https://github.com/ValRCS/Digital_Discourse_ETH713_Fall_2020/blob/main/TextProcessing/CountingWords_2026.ipynb) · [Week 4 — Using_JSON_APIs_2026](https://github.com/ValRCS/Digital_Discourse_ETH713_Fall_2020/blob/main/JSON_APIs/Using_JSON_APIs_2026.ipynb).


## 1 · From JSON APIs to HTML (5 minutes)

Both JSON and HTML are textual formats. HTTP transfers bytes; a client decodes them to text. A parser recognizes the syntax and builds a structured representation. JSON is primarily a data interchange format, while HTML is a markup language for documents and interfaces. A JSON API may return HTML, and a web page may contain JSON; the distinction is about format, not a strict division between URLs.

| Concept | JSON | HTML |
|---|---|---|
| Raw text | Python `str` | Python `str` |
| Parser | `json.loads(text)` | `BeautifulSoup(text, 'html.parser')` |
| Parsed structure | dict/list/scalar | tree of tags and text nodes |
| Navigation | keys and indices | elements, attributes, selectors |
| Extraction | access values | choose and extract elements |
| Output | Python records | Python records |

**Parsing is not downloading.** The same HTML string can be parsed offline; after extraction, we can serialize the resulting Python records as JSON even though the original source was HTML.


### Setup

Colab normally provides these packages. If imports fail locally, run `%pip install requests beautifulsoup4 pandas lxml` in a separate cell, restart the kernel if needed, then run all. The Python standard library supplies the other imports. Outputs go to `outputs/web_scraping_2026/` relative to the current working directory.


In [ ]:
import json
import re
import sys
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin, urlsplit

import requests
import bs4
from bs4 import BeautifulSoup
import pandas as pd

output_dir = Path("outputs/web_scraping_2026")
output_dir.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0], "| BeautifulSoup:", bs4.__version__, "| pandas:", pd.__version__)


## 2 · HTML as a structured document (10 minutes)

An HTML **tag** is markup such as `<p>` or `</p>`. An **element** includes the tag(s), attributes and content. An **attribute** such as `class="news"` or `href="/articles/123"` provides metadata or a link. Text nodes hold actual textual content. An element may contain other elements (children and descendants); its containing element is its parent.

A browser constructs a Document Object Model (DOM) from HTML and may later modify it with JavaScript. BeautifulSoup constructs its own parse tree from the supplied markup, not the browser's live DOM. Inspect a webpage using View Source, Developer Tools → Elements, and Network. These views can differ.


In [ ]:
ARTICLE_HTML = "<!doctype html>\n<html lang=\"lv\"><head><meta charset=\"utf-8\"><title>Teaching articles</title></head>\n<body><nav>Izvēlne</nav><main>\n<article class=\"news\"><h2>Digitālās humanitārās zinātnes Rīgā</h2><p>Pētnieki analizē digitālus tekstus.</p><a href=\"/articles/123\">Lasīt vairāk</a></article>\n<article class=\"news\"><h2>Bibliotēka un valoda</h2><p>Bibliotēka saglabā kultūras mantojumu.</p></article>\n</main><footer>Kontaktinformācija</footer></body></html>"
print(type(ARTICLE_HTML).__name__)
print(ARTICLE_HTML)


### From markup to tree

For the first article, the simplified tree is:

    article [class=news]
    ├── h2 → heading text
    ├── p  → paragraph text
    └── a  [href=/articles/123] → link text

The attribute is not the visible link text. The document contains navigation and a footer as well as articles. Selecting the relevant elements is part of defining the corpus.


In [ ]:
article_soup = BeautifulSoup(ARTICLE_HTML, "html.parser")
print(type(article_soup).__name__)
print(article_soup.title.get_text(strip=True))
first_article = article_soup.find("article")
print(first_article.prettify())


## 3 · BeautifulSoup fundamentals (15 minutes)

| Operation | Result |
|---|---|
| `find('h2')` | first matching tag or None |
| `find_all('p')` | list-like collection of matching tags |
| `select_one('article h2')` | first CSS selector match or None |
| `select('article p')` | all CSS selector matches |
| `tag.get_text(' ', strip=True)` | text with whitespace cleaned at text-node boundaries |
| `tag.get('href')` | attribute value or None |

CSS selectors: `p` means a tag; `.news` means a class; `#listings` means an ID; `article p` means a descendant; `article > p` means a direct child. Selectors express structural assumptions, which may fail when a site changes.


In [ ]:
print(article_soup.find("h2").get_text(" ", strip=True))
print([p.get_text(" ", strip=True) for p in article_soup.find_all("p")])
print([h.get_text(" ", strip=True) for h in article_soup.select("article.news > h2")])
print(article_soup.select_one(".missing"))


### Attributes and relative URLs

`href` can contain an absolute URL or a relative path. Resolve relative paths against the actual document URL with `urljoin`, rather than manually concatenating strings. The domain example.invalid is reserved for examples and is not a real data source.


In [ ]:
link = first_article.select_one("a")
base_url = "https://example.invalid/teaching/"
print("Text:", link.get_text(" ", strip=True))
print("Raw href:", link.get("href"))
print("Resolved:", urljoin(base_url, link.get("href")))


### Scope matters: document text versus research text

`get_text()` on the entire document can include menus, footers and other boilerplate. Extract the relevant elements instead. The separator prevents adjacent text nodes from being accidentally merged. A missing tag is not the same as an empty string.


In [ ]:
def text_or_none(tag):
    if tag is None:
        return None
    return tag.get_text(" ", strip=True) or None

print("Whole document:", article_soup.get_text(" ", strip=True))
for article in article_soup.select("article.news"):
    heading = text_or_none(article.select_one("h2"))
    paragraph = text_or_none(article.select_one("p"))
    print(heading, "|", paragraph)
assert article_soup.select_one(".missing") is None
assert len(article_soup.select("article.news")) == 2


## 4 · HTTP retrieval and responsible scraping (10 minutes)

A GET request asks a server for a resource. `response.content` holds bytes; `response.text` holds decoded text; `response.json()` parses JSON, not HTML. Inspect status, Content-Type and final URL (redirects may occur). A 200 response does not prove the desired content is present. 403/429 responses are reasons to stop, not to circumvent controls. Timeouts limit waiting for a server; they are not a complete overall runtime limit.

Before collecting data, prefer an API or download, inspect site terms and robots.txt, and consider copyright, privacy and research ethics. Robots.txt is a crawler policy, not blanket legal permission or prohibition. Do not bypass login, CAPTCHAs or restrictions. Use few requests, avoid collecting unnecessary personal data, and preserve provenance. Do not assume a live site is suitable for bulk extraction. This lesson uses the deliberately provided Quotes to Scrape sandbox for an optional single-page request.


In [ ]:
def fetch_html(url):
    response = requests.get(
        url, timeout=(5, 15),
        headers={"User-Agent": "DE0823-educational-demo/2026"}
    )
    response.raise_for_status()
    media_type = response.headers.get("Content-Type", "").split(";")[0].strip().lower()
    if media_type not in {"text/html", "application/xhtml+xml"}:
        raise ValueError(f"Unexpected content type: {media_type or 'missing'}")
    return response.text, response.url, datetime.now(timezone.utc).isoformat()

def parse_html(html):
    return BeautifulSoup(html, "html.parser")


### Live mode is optional

Set `USE_LIVE = True` only for the sandbox demonstration after checking its policies. The default is offline and reproducible. One request is made; if it fails or does not return the expected page structure, the code uses synthetic data and explicitly labels the fallback. In a real research pipeline, failures should be recorded and never silently substituted with invented evidence.


In [ ]:
QUOTE_HTML = "<!doctype html>\n<html lang=\"en\"><head><meta charset=\"utf-8\"><title>Synthetic quotes for teaching</title></head>\n<body><main>\n<div class=\"quote\"><span class=\"text\">Words shape how we understand the world.</span><small class=\"author\">Example Author A</small><a href=\"/author/a\">About</a><div class=\"tags\"><a class=\"tag\">language</a><a class=\"tag\">research</a></div></div>\n<div class=\"quote\"><span class=\"text\">Libraries preserve words and cultural memory.</span><small class=\"author\">Example Author B</small><a href=\"/author/b\">About</a><div class=\"tags\"><a class=\"tag\">memory</a><a class=\"tag\">language</a></div></div>\n<div class=\"quote\"><span class=\"text\">Rīga is a place of many voices.</span><small class=\"author\">Example Author A</small><a href=\"/author/a\">About</a><div class=\"tags\"><a class=\"tag\">culture</a></div></div>\n<div class=\"quote\"><span class=\"text\">Research begins with a question about words.</span><small class=\"author\">Example Author C</small><div class=\"tags\"></div></div>\n<div class=\"quote\"><span class=\"text\">Incomplete record for testing.</span></div>\n</main></body></html>"
USE_LIVE = False
QUOTE_URL = "https://quotes.toscrape.com/"
html = QUOTE_HTML
document_url = None
retrieved_at_utc = None
source_kind = "synthetic_offline"
fallback_reason = None

if USE_LIVE:
    try:
        candidate, final_url, timestamp = fetch_html(QUOTE_URL)
        if not parse_html(candidate).select(".quote .text"):
            raise ValueError("Expected quotation elements not found")
        html, document_url, retrieved_at_utc = candidate, final_url, timestamp
        source_kind = "live_sandbox"
    except (requests.RequestException, ValueError) as exc:
        source_kind = "synthetic_fallback"
        fallback_reason = f"{type(exc).__name__}: {exc}"

soup = parse_html(html)
print("Source:", source_kind, "| URL:", document_url)
if fallback_reason:
    print("Live retrieval failed:", fallback_reason)
print("Characters:", len(html), "| quote elements:", len(soup.select(".quote")))


## 5 · Build a textual dataset (15 minutes)

[Quotes to Scrape](https://quotes.toscrape.com/) is a practice site with repeated quotation blocks, attributed authors and tags. Its attributions and tags are website claims, not verified evidence of authorship. Our offline fixture contains entirely invented texts and authors with one deliberately incomplete record. It also contains a valid record without a link or tags.

First inspect a single element; then extract one record; then use a loop. Avoid selecting all authors and all quotes independently and zipping them: missing fields can misalign records.


In [ ]:
quote_elements = soup.select(".quote")
if not quote_elements:
    raise ValueError("No quotation blocks found")
quote = quote_elements[0]
print(quote.prettify())


In [ ]:
first_record = {
    "text": text_or_none(quote.select_one(".text")),
    "author": text_or_none(quote.select_one(".author")),
    "tags": [tag.get_text(" ", strip=True) for tag in quote.select(".tag")],
}
print(first_record)


### From one record to a reusable extraction function

Text and attributed author are required in this particular dataset. Tags and author links are optional. Missing required fields are counted; missing optional links become None, and no tags becomes an empty list. Keep the original Unicode text. Only resolve HTTP(S) links from live documents; synthetic links remain unresolved examples.


In [ ]:
def extract_quote(element, document_url=None):
    text = text_or_none(element.select_one(".text"))
    author = text_or_none(element.select_one(".author"))
    if not text or not author:
        return None
    tags = [value for tag in element.select(".tag")
            if (value := text_or_none(tag))]
    anchor = element.select_one("a[href^='/author/'], a[href*='/author/']")
    href = anchor.get("href") if anchor else None
    author_url = urljoin(document_url, href) if document_url and href else None
    if author_url and urlsplit(author_url).scheme not in {"http", "https"}:
        author_url = None
    return {
        "text": text, "author": author, "tags": tags,
        "author_href": href, "author_url": author_url,
        "source_url": document_url
    }

records = []
skipped = 0
for element in quote_elements:
    record = extract_quote(element, document_url)
    if record is None:
        skipped += 1
    else:
        records.append(record)

if not records:
    raise ValueError("No valid quotation records extracted")
print("Valid:", len(records), "| Skipped:", skipped)
print(records[0])


In [ ]:
assert len(records) + skipped == len(quote_elements)
assert all(record["text"] and record["author"] for record in records)
if source_kind != "live_sandbox":
    assert len(records) == 4 and skipped == 1
    assert all(record["source_url"] is None for record in records)
    assert records[-1]["tags"] == [] and records[-1]["author_url"] is None
df = pd.DataFrame(records)
df


### Provenance and serialization

A list of dictionaries becomes a DataFrame for inspection or a JSON document for exchange. In JSON, arrays can preserve the list of tags. CSV is flat, so explicitly serialize that column as JSON text. Store provenance separately for CSV users and alongside records in JSON. The retrieval timestamp is not a publication date. Synthetic records have no real source URL or retrieval timestamp. Dataset creation time and live retrieval time are different concepts.


In [ ]:
metadata = {
    "course": "DE0823", "source_kind": source_kind,
    "requested_url": QUOTE_URL if USE_LIVE else None,
    "document_url": document_url,
    "retrieved_at_utc": retrieved_at_utc,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection": "one page; .quote blocks; require text and author",
    "fixture": "fixtures/sample_quotes.html" if source_kind != "live_sandbox" else None,
    "fallback_reason": fallback_reason,
    "elements_found": len(quote_elements),
    "records_extracted": len(records),
    "records_skipped": skipped
}
dataset = {"metadata": metadata, "records": records}
json_path = output_dir / "quotes.json"
json_path.write_text(json.dumps(dataset, ensure_ascii=False, indent=2), encoding="utf-8")
csv_path = output_dir / "quotes.csv"
csv_df = df.copy()
csv_df["tags"] = csv_df["tags"].map(lambda tags: json.dumps(tags, ensure_ascii=False))
csv_df.to_csv(csv_path, index=False, encoding="utf-8")
(output_dir / "quotes_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(json_path, csv_path, sep="\n")


## 6 · HTML tables with pandas (10 minutes)

`pandas.read_html` extracts HTML tables and returns a list of DataFrames. It cannot generally recover all hyperlinks, attributes or non-tabular text by default. Pass an HTML string via `StringIO`; a URL would trigger a network request. Avoid assuming that the fifth table always holds the data. Select by an ID or identify expected columns and check uniqueness.


In [ ]:
TABLE_HTML = "<!doctype html>\n<html lang=\"lv\"><head><meta charset=\"utf-8\"><title>Synthetic listings</title></head><body>\n<table><tr><th>Navigation</th></tr><tr><td>Home</td></tr></table>\n<table id=\"listings\"><thead><tr><th>Street</th><th>Rooms</th><th>Area m2</th><th>Price EUR</th><th>Description</th></tr></thead>\n<tbody><tr><td>Example 1</td><td>2</td><td>48</td><td>85000</td><td>Gaišs un mājīgs dzīvoklis</td></tr>\n<tr><td>Example 2</td><td>3</td><td>70</td><td>125000</td><td>Plašs dzīvoklis parkam tuvumā</td></tr>\n<tr><td>Example 3</td><td>1</td><td>32</td><td>59000</td><td>Silts un renovēts dzīvoklis</td></tr></tbody></table>\n</body></html>"
frames = pd.read_html(StringIO(TABLE_HTML))
for index, frame in enumerate(frames):
    print(index, frame.shape, list(frame.columns))

expected_columns = {"Street", "Rooms", "Area m2", "Price EUR", "Description"}
matches = [frame for frame in frames if expected_columns.issubset(frame.columns)]
if len(matches) != 1:
    raise ValueError(f"Expected one matching table, found {len(matches)}")
listings = matches[0].copy()
assert len(listings) == 3
listings


In [ ]:
listings["Price per m2"] = listings["Price EUR"] / listings["Area m2"]
listings.to_csv(output_dir / "synthetic_listings.csv", index=False, encoding="utf-8")
listings


### Authentic Latvian commercial discourse: SS.com

The older notebooks scrape [Āgenskalns apartment listings](https://www.ss.com/en/real-estate/flats/riga/agenskalns/sell/). They historically used `dfs[4]`, positional cells and repeated requests; website layouts and policies can change. Do not make SS.com requests in this core notebook. The repository contains a historical CSV, JSON and XLSX snapshot. Use the archived CSV for analysis if you have a local copy, or download it manually from the repository when needed. It includes truncated descriptions, not necessarily full advertisements. The synthetic table above is not a representation of current prices.


In [ ]:
archive_candidates = [Path("agenskalns.csv"), Path("WebScraping/agenskalns.csv")]
archive_path = next((path for path in archive_candidates if path.is_file()), None)
if archive_path:
    archive_df = pd.read_csv(archive_path, encoding="utf-8")
    print("Historical archive:", archive_path, "|", archive_df.shape)
    print(archive_df.columns.tolist())
    print(archive_df.head(2).to_string(index=False))
else:
    print("Optional archive not present. Core lesson continues with synthetic data.")


## 7 · From scraped text to discourse analysis (10 minutes)

Revisit Week 3: Unicode NFC normalization, case folding and tokenization, then count words using Counter. NFC makes equivalent Unicode representations comparable; case folding reduces case distinctions. This simple tokenizer extracts letter sequences, splitting hyphenated forms and not performing lemmatization. Word frequencies measure forms, not meaning.


In [ ]:
def tokenize(text):
    normalized = unicodedata.normalize("NFC", text).casefold()
    return re.findall(r"[^\W\d_]+", normalized, flags=re.UNICODE)

tokens = [word for record in records for word in tokenize(record["text"])]
word_counts = Counter(tokens)
tag_counts = Counter(tag for record in records for tag in record["tags"])
author_counts = Counter(record["author"] for record in records)
print("Source:", source_kind)
print("Token count:", len(tokens))
print("Most common words:", word_counts.most_common(10))
print("Tags:", tag_counts.most_common())
print("Attributed authors:", author_counts.most_common())


In [ ]:
total_tokens = len(tokens)
frequency_df = pd.DataFrame([
    {"word": word, "count": count, "per_1000_tokens": 1000 * count / total_tokens}
    for word, count in word_counts.most_common()
], columns=["word", "count", "per_1000_tokens"])
frequency_df.to_csv(output_dir / "word_frequencies.tsv", sep="\t", index=False, encoding="utf-8")
frequency_df.head(10)


### Interpret results with caution

What is included or excluded by our selectors? Tags are website classifications rather than independent linguistic evidence. Authors may be misattributed. One page is a convenience sample, not a representative corpus. Synthetic outputs are suitable for checking code, not for empirical claims. In Latvian commercial discourse, words such as mājīgs, silts, plašs and ekskluzīvs could be investigated as persuasive descriptors; actual counts require an appropriately defined corpus.


In [ ]:
assert sum(word_counts.values()) == len(tokens)
assert tokenize("RĪGA Ri\u0304ga") == ["rīga", "rīga"]
if source_kind != "live_sandbox":
    assert word_counts["words"] == 3
    assert tag_counts["language"] == 2
    assert author_counts["Example Author A"] == 2


## 8 · Student exercises and validation (15 minutes)

**A (4 minutes):** Select just the article headings and then the paragraphs. What changes if you select the whole body? Extract the href of the first link.

**B (4 minutes):** Modify a copy of the synthetic quotation HTML: add a record with text and author but no tags. Then remove its author. Predict and verify how many records are skipped.

**C (4 minutes):** Compute the frequency of quotation tags and the relative frequency per 1,000 tokens of a word of your choice. Explain why tag frequencies and word frequencies measure different things.

**D (3 minutes):** Inspect the JSON and CSV exports. Why is the tags column encoded differently? Identify the source kind and explain what conclusions the data supports.


In [ ]:
# Exercise workspace: modify a copy without changing the core dataset.
practice_soup = parse_html(ARTICLE_HTML)
print([text_or_none(tag) for tag in practice_soup.select("article h2")])
# Add your own selector or extraction code below.


### Check the saved data

Read the JSON and CSV back, check record counts, nested tag lists, Unicode preservation and provenance. Assertions are inexpensive regression checks, not proof of research validity.


In [ ]:
saved = json.loads(json_path.read_text(encoding="utf-8"))
saved_csv = pd.read_csv(csv_path, encoding="utf-8", keep_default_na=False)
assert saved == dataset
assert len(saved_csv) == len(records)
assert saved_csv["tags"].map(json.loads).tolist() == [r["tags"] for r in records]
assert saved_csv["text"].tolist() == [r["text"] for r in records]
assert saved["metadata"]["source_kind"] in {"synthetic_offline", "synthetic_fallback", "live_sandbox"}
assert len(pd.read_csv(output_dir / "word_frequencies.tsv", sep="\t")) == len(word_counts)
assert pd.read_csv(output_dir / "synthetic_listings.csv")["Description"].tolist() == listings["Description"].tolist()
print("Validation passed.")
print("Outputs:", sorted(path.name for path in output_dir.iterdir()))


## Optional extension A · Pagination without a network

Pagination means repeated pages of a collection. Do not guess the number of pages or infer it from `rel=prev` on arbitrary websites. Identify an explicit next link, resolve it against the current document URL, keep a visited set, impose a small maximum, and stop at a missing link, repeated URL or unexpected origin. For live research also check policies, rate limits and failures, avoid automatic parallel requests, and cache responses. The following exercise uses a synthetic two-page site with a deliberate cycle; no request is sent.


In [ ]:
PAGES = {
    "https://example.invalid/page/1": '<p>First page</p><a rel="next" href="/page/2">Next</a>',
    "https://example.invalid/page/2": '<p>Second page</p><a rel="next" href="/page/1">Next</a>'
}
def walk_fixture_pages(start, max_pages=3):
    visited = set()
    current = start
    results = []
    while current and len(results) < max_pages and current not in visited:
        if current not in PAGES:
            break
        visited.add(current)
        page = parse_html(PAGES[current])
        results.append(current)
        next_link = page.select_one("a[rel~=next][href]")
        current = urljoin(current, next_link["href"]) if next_link else None
    return results

visited_pages = walk_fixture_pages("https://example.invalid/page/1")
assert len(visited_pages) == 2
print(visited_pages)


## Optional extension B · Dynamic websites

The initial HTTP response may contain little content if JavaScript later retrieves data or modifies the DOM. Use browser Developer Tools → Network and Elements to understand the difference. Prefer documented APIs where available; do not circumvent access controls. Selenium and Playwright automate browsers but are heavier than requests and BeautifulSoup; Scrapy supports larger crawling workflows. No browser automation or driver setup is needed today. The historical Selenium notebook contains obsolete locator calls; consult the current documentation instead.


## Optional extension C · Download or reuse outputs

In Colab, run the following in a separate cell if you want to download the JSON file:

```python
from google.colab import files
files.download(str(json_path))
```

For local Jupyter, files are already in the output directory. No Google Drive mount is necessary. Standalone synthetic fixtures identical to the embedded examples are available in the [fixtures directory](https://github.com/ValRCS/Digital_Discourse_ETH713_Fall_2020/tree/main/WebScraping/fixtures).


## Further reading and instructor notes

- [Beautiful Soup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [MDN HTML](https://developer.mozilla.org/en-US/docs/Web/HTML) and [HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP)
- [Requests documentation](https://requests.readthedocs.io/en/latest/)
- [pandas.read_html](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html)
- [Quotes to Scrape](https://quotes.toscrape.com/)
- [Selenium documentation](https://www.selenium.dev/documentation/)
- [Web scraping directory and historical notebooks](https://github.com/ValRCS/Digital_Discourse_ETH713_Fall_2020/tree/main/WebScraping)

**Teaching takeaway:** research question → source selection → retrieval → parsing → extraction → corpus construction → analysis → interpretation. Each step introduces methodological decisions.
